In [2]:
import psycopg2
import pandas as pd
import numpy as np
from datetime import datetime


In [9]:

DB_CONFIG = {
    "host": "172.23.106.87",
    "port": 5434,
    "database": "thingsboard",
    "user": "tbuser",
    "password": "tbpassword"
}

conn = psycopg2.connect(**DB_CONFIG)
print("Connected to TimescaleDB")


Connected to TimescaleDB


In [10]:
START_TIME = "2025-12-14 00:00:00"
END_TIME   = "2025-12-14 23:59:59"

RACKS = ["RackA_Device", "RackB_Device", "RackC_Device", "RackD_Device"]


In [24]:
query = """
SELECT
    d.name AS device_name,
    to_timestamp(t.ts / 1000.0) AT TIME ZONE 'UTC' AS time,
    t.key,
    CASE
        WHEN t.bool_v IS NOT NULL THEN
            CASE WHEN t.bool_v THEN 1.0 ELSE 0.0 END
        WHEN t.long_v IS NOT NULL THEN t.long_v::double precision
        WHEN t.dbl_v  IS NOT NULL THEN t.dbl_v
    END AS value
FROM ts_kv t
JOIN device d ON d.id = t.entity_id
WHERE d.name = ANY(%(racks)s)
  AND to_timestamp(t.ts / 1000.0) BETWEEN
        %(start_time)s::timestamp
    AND %(end_time)s::timestamp
ORDER BY time;
"""


In [25]:
params = {
    "racks": RACKS,              # list[str]
    "start_time": START_TIME,    # datetime
    "end_time": END_TIME         # datetime
}

df_raw = pd.read_sql(query, conn, params=params)


C:\Users\Admin\AppData\Local\Temp\ipykernel_21820\2723555957.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(query, conn, params=params)


In [28]:
df = (
    df_raw
    .pivot_table(
        index=["time", "device_name"],
        columns="key",
        values="value"
    )
    .reset_index()
    .sort_values("time")
)


In [31]:
df.head(50)


key,time,rack_id,34,64,65,66,67,68,69
0,2025-12-14 07:17:20.685,RackA_Device,46.52,79.38,1.29,25.29,29.26,34.92,4382.0
1,2025-12-14 07:18:28.804,RackA_Device,45.84,38.56,0.89,25.90,27.83,33.55,3157.0
2,2025-12-14 07:18:38.785,RackA_Device,48.12,76.35,1.26,25.49,29.30,34.78,4290.0
3,2025-12-14 07:18:48.804,RackA_Device,49.08,45.92,0.96,25.47,27.76,33.31,3378.0
4,2025-12-14 07:18:59.007,RackA_Device,49.78,40.66,0.91,25.62,27.66,33.43,3220.0
5,2025-12-14 07:19:08.829,RackA_Device,46.40,37.88,0.88,25.50,27.39,33.34,3136.0
6,2025-12-14 07:19:18.824,RackA_Device,49.99,61.08,1.11,25.26,28.31,33.57,3832.0
7,2025-12-14 07:19:28.834,RackA_Device,48.34,35.04,0.85,25.11,26.86,32.34,3051.0
8,2025-12-14 07:19:38.816,RackA_Device,49.11,65.39,1.15,25.00,28.27,33.34,3962.0
9,2025-12-14 07:19:48.829,RackA_Device,49.60,46.39,0.96,25.29,27.61,32.77,3392.0
